# Battery model and energy accounting

Defines the battery parameters and energy accounting (power, capacity, efficiency, state of charge) used throughout the project, with a worked charge/discharge example.


In [1]:
# Convert power over a time interval into energy.
power_mw = 1.0
time_hours = 15 / 60
energy_mwh = power_mw * time_hours

print(f"Energy bought in 15 minutes: {energy_mwh:.2f} MWh")


Energy bought in 15 minutes: 0.25 MWh


In [2]:
# Unit suffixes make calculations and validation easier to audit.
capacity_mwh = 2.0
power_limit_mw = 1.0
minimum_soc_mwh = 0.10 * capacity_mwh
maximum_soc_mwh = 0.90 * capacity_mwh
starting_soc_mwh = 0.8
charge_efficiency = 0.92
discharge_efficiency = 0.92
step_hours = 1.0
wear_eur_per_mwh_discharged = 5.0

round_trip_efficiency = charge_efficiency * discharge_efficiency

print(f"Allowed SoC range: {minimum_soc_mwh:.1f} to {maximum_soc_mwh:.1f} MWh")
print(f"Round-trip efficiency: {round_trip_efficiency:.2%}")


Allowed SoC range: 0.2 to 1.8 MWh
Round-trip efficiency: 84.64%


## charge at a low price


In [3]:
buy_price_eur_per_mwh = 40.0
charge_power_mw = 1.0

# Energy bought is measured at the grid connection.
grid_energy_bought_mwh = charge_power_mw * step_hours

# Apply charging efficiency when energy enters the battery.
soc_after_charge_mwh = starting_soc_mwh + charge_efficiency * grid_energy_bought_mwh
purchase_cost_eur = buy_price_eur_per_mwh * grid_energy_bought_mwh

print(f"Grid energy bought: {grid_energy_bought_mwh:.4f} MWh")
print(f"SoC after charging: {soc_after_charge_mwh:.4f} MWh")
print(f"Purchase cost: {purchase_cost_eur:.2f} EUR")


Grid energy bought: 1.0000 MWh
SoC after charging: 1.7200 MWh
Purchase cost: 40.00 EUR


## discharge at a high price, fairly


In [4]:
sell_price_eur_per_mwh = 100.0

# Discharge only the energy added during charging, after discharge losses.
energy_removed_from_battery_mwh = soc_after_charge_mwh - starting_soc_mwh
grid_energy_sold_mwh = energy_removed_from_battery_mwh * discharge_efficiency
discharge_power_mw = grid_energy_sold_mwh / step_hours

# Confirm that terminal state of charge matches the initial state.
ending_soc_mwh = soc_after_charge_mwh - grid_energy_sold_mwh / discharge_efficiency

sales_revenue_eur = sell_price_eur_per_mwh * grid_energy_sold_mwh
gross_trading_margin_eur = sales_revenue_eur - purchase_cost_eur
wear_cost_eur = wear_eur_per_mwh_discharged * grid_energy_sold_mwh
contribution_after_modelled_wear_eur = gross_trading_margin_eur - wear_cost_eur

print(f"Grid energy sold: {grid_energy_sold_mwh:.4f} MWh")
print(f"Ending SoC: {ending_soc_mwh:.4f} MWh")
print(f"Sales revenue: {sales_revenue_eur:.2f} EUR")
print(f"Gross trading margin: {gross_trading_margin_eur:.2f} EUR")
print(f"Modelled wear allowance: {wear_cost_eur:.2f} EUR")
print(f"Contribution after modelled wear: {contribution_after_modelled_wear_eur:.2f} EUR")

# This excludes market fees, tariffs, taxes, capex and other real-world costs.


Grid energy sold: 0.8464 MWh
Ending SoC: 0.8000 MWh
Sales revenue: 84.64 EUR
Gross trading margin: 44.64 EUR
Modelled wear allowance: 4.23 EUR
Contribution after modelled wear: 40.41 EUR


## Check the calculation


In [5]:
import math

# Validate power and state-of-charge limits.
assert 0 <= charge_power_mw <= power_limit_mw
assert 0 <= discharge_power_mw <= power_limit_mw
assert minimum_soc_mwh <= starting_soc_mwh <= maximum_soc_mwh
assert minimum_soc_mwh <= soc_after_charge_mwh <= maximum_soc_mwh
assert minimum_soc_mwh <= ending_soc_mwh <= maximum_soc_mwh

# Ensure that the comparison does not use free initial battery inventory.
assert math.isclose(ending_soc_mwh, starting_soc_mwh, abs_tol=1e-9)

# Validate known values for this deterministic example.
assert math.isclose(grid_energy_sold_mwh, 0.8464, abs_tol=1e-9)
assert math.isclose(contribution_after_modelled_wear_eur, 40.408, abs_tol=1e-9)

print("All battery accounting checks passed.")


All battery accounting checks passed.
